# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">3주차 · 같은 추천 방식을 다르게 채점해 보고 어느 점수를 믿을지 정하기</mark>

지난 두 주 동안 점수를 올렸습니다. 1주차 0.2163, 2주차 0.2684 였습니다.

오늘은 **추천 방식을 한 글자도 고치지 않습니다.** 2주차에서 만든 세그먼트 추천을 그대로 가져옵니다.

대신 **재는 방법**을 바꿔 봅니다. 그것만으로 점수가 달라지는 것을 직접 확인합니다.

---

### 오늘의 구성

| 파트 | 종류 | 어디서 | 하는 일 |
|---|---|---|---|
| 1 | 개념 | 슬라이드 | 데이터를 가리는 방법이 두 가지라는 것 |
| 2 | 실습 | **이 노트북 3.2~3.5** | 같은 추천 방식을 두 방법으로 채점해 차이를 본다 |
| 3 | 개념 | 슬라이드 | 데이터 누수 · 콜드 스타트 |
| 4 | 실습 | **이 노트북 3.6~3.8** | 신규 투자자만 따로 채점하고 무엇을 줄지 정한다 |

이 노트북은 **실습 두 파트**만 담고 있습니다. 그래서 절 번호가 `3.` 부터 시작합니다.


## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

이 노트북은 **아무것도 설치하지 않고** 브라우저에서 바로 실행할 수 있습니다.

### Google Colab 으로 열기

1. 아래 주소를 눌러 주세요.
   - https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/03_split_leakage.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 "실습 준비" 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다
import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")


내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3 · 실습 — 같은 추천 방식을 두 방법으로 채점한다</mark>

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.1 오늘 나오는 말  `[PPT]`</mark>

1. **홀드아웃 (holdout)** — 채점하려고 일부러 가려 두는 몫
   - 우리는 **마지막 한 달**을 가려 뒀습니다. 그 한 달이 정답지입니다.
2. **무작위 분할 (random split)** — 기록을 섞어서 아무 데서나 떼는 것
   - 가장 흔히 쓰이고 코드도 한 줄입니다. 오늘 직접 만들어 봅니다.
3. **시간순 분할 (time-based split)** — 마지막 기간을 통째로 떼는 것
   - 우리가 1주차부터 써 온 방법입니다. `recsys.split_by_time` 이 이것입니다.
4. **데이터 누수 (data leakage)** — 배우면 안 되는 정보가 새어 들어가 점수를 부풀리는 것
   - 오늘 그것을 **일부러 한 번 만들어 봅니다.**
5. **콜드 스타트 (cold start)** — 기록이 없어 취향을 짐작할 재료가 없는 상태
   - 이번 달에 처음 온 사람이 그렇습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.2 준비 — 2주차 추천 방식을 그대로 되살린다</mark>

**3.2 에서 하는 일** — 2주차에서 만든 세그먼트 추천을 그대로 다시 만듭니다.

**1. 왜 그대로 가져오나**
- 오늘 바꾸는 것은 **재는 방법 하나**입니다. 추천 방식이 달라지면 무엇 때문에 점수가 변했는지 알 수 없습니다.
- 그래서 세그먼트 나누는 법도, 순위표 만드는 법도 지난주와 **한 글자도 다르지 않게** 둡니다.

**2. 무엇을 만드나**
- 표 세 개를 읽고, 종목의 섹터·위험도 사전을 만듭니다.
- 사람을 세그먼트로 나누는 **함수**와, 세그먼트마다 순위표를 만드는 **함수**를 만듭니다.
- 지난주에는 코드로 쭉 썼는데, 오늘은 **두 번 쓸 것이라 함수로 묶습니다.**

**3. 설명 없이 한 번에 실행합니다** — 지난주에 이미 한 것이라 오늘은 결과만 확인합니다.


In [2]:
# 이 셀에서 하는 일 — 오늘 쓸 도구를 불러온다
# 왜 하나 — pandas 로 표를 다루고, recsys.py 에는 1·2주차에서 만든 함수가 들어 있다
import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구

sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 칸 안쪽 폴더도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. 앞으로 pd 라고 부른다
import recsys                               # 이 수업용으로 만든 도구 모음

print("도구 준비 완료 · pandas", pd.__version__)


도구 준비 완료 · pandas 3.0.5


In [3]:
# 이 셀에서 하는 일 — 데이터 세 표를 읽고, 번호를 이름·섹터·위험도로 바꿔 줄 사전 3개를 만든다
# 왜 하나 — 거래 기록에는 I079 같은 번호만 있다. 번호만 보면 무슨 종목인지 알 수 없어서 미리 만들어 둔다
items, users, interactions = recsys.load()          # 종목·투자자·거래 기록 세 표
print(f"종목 {len(items)}개 · 투자자 {len(users)}명 · 거래 기록 {len(interactions):,}건")

종목표_번호색인 = items.set_index("item_id")         # 종목 번호를 행 이름으로 세운다
이름_사전 = 종목표_번호색인["name"].to_dict()        # {번호: 이름}
섹터_사전 = 종목표_번호색인["sector"].to_dict()      # {번호: 섹터}
위험도_사전 = 종목표_번호색인["risk_level"].to_dict()  # {번호: 1~5 숫자}
print(f"사전 3개 준비 ·  I079 → {이름_사전['I079']} · {섹터_사전['I079']} · 위험도 {위험도_사전['I079']}")


종목 100개 · 투자자 300명 · 거래 기록 8,086건
사전 3개 준비 ·  I079 → KODEX 레버리지 · 레버리지 · 위험도 5


In [4]:
# 이 셀에서 하는 일 — 2주차의 세그먼트 나누기·순위표 만들기를 함수 2개로 옮겨 놓는다
# 왜 하나 — 오늘 채점을 네 번 하는데 그때마다 이 두 가지가 필요하다. 추천 방식은 2주차 그대로 — 오늘 바꾸는 것은 채점 방법뿐이다
import collections                                   # 개수를 세어 주는 도구가 들어 있다


def 세그먼트_나누기(학습구간_기록):
    """사람마다 주력 섹터와 평균 위험도를 뽑아 세그먼트 이름을 붙인다. 2주차와 같다."""
    무리 = {}                                         # {사람: 세그먼트 이름}
    for 사람, 그_사람의_기록 in 학습구간_기록.groupby("user_id"):   # 사람별로 묶어서 한 명씩
        섹터_세기 = collections.Counter()             # 섹터가 몇 번씩 나왔는지 셀 그릇
        위험도_합 = 0                                 # 위험도를 더해 갈 그릇
        for 번호 in 그_사람의_기록["item_id"]:         # 그 사람이 담은 종목을 하나씩
            섹터_세기[섹터_사전[번호]] += 1            # 그 종목의 섹터를 한 번 센다
            위험도_합 = 위험도_합 + 위험도_사전[번호]   # 위험도를 더한다
        주력_섹터 = 섹터_세기.most_common(1)[0][0]     # 가장 많이 나온 섹터 하나
        평균_위험도 = round(위험도_합 / len(그_사람의_기록))   # 평균을 내어 반올림
        무리[사람] = f"{주력_섹터}|{평균_위험도}"       # 둘을 파이프로 이어 이름 하나로
    return 무리


def 순위표_만들기(학습구간_기록, 무리):
    """세그먼트마다 그 안에서만 세어 인기 순위표를 만든다. 2주차와 같다."""
    세그먼트_열 = 학습구간_기록["user_id"].map(무리)      # 사람 아이디 → 세그먼트 이름
    거래_세그먼트포함 = 학습구간_기록.assign(group=세그먼트_열)   # group 열을 더한 새 표
    순위표 = {}                                        # {세그먼트: 종목 순위 목록}
    for 이름, 그_거래 in 거래_세그먼트포함.groupby("group"):     # 세그먼트마다 한 번씩
        순위표[이름] = list(그_거래["item_id"].value_counts().index)   # 그 안에서만 센다
    return 순위표


# 만들었으면 한 번 돌려 봅니다. 여기서는 전체 기록을 넣어 모양만 봅니다.
맛보기_무리 = 세그먼트_나누기(interactions)              # {사람: 세그먼트 이름} 이 나온다
세그먼트_이름들 = sorted(set(맛보기_무리.values()))       # set = 중복을 없앤 묶음, sorted = 가나다순
print(f"세그먼트 나누기 → 사람 {len(맛보기_무리)}명이 세그먼트 {len(세그먼트_이름들)}가지로 묶인다")
print("  이름 예시 :", 세그먼트_이름들[:3], "  ← 주력 섹터|평균 위험도 를 이어 붙인 이름")

맛보기_순위표 = 순위표_만들기(interactions, 맛보기_무리)   # {세그먼트: 종목 번호 순위}
첫_세그먼트 = 세그먼트_이름들[0]                          # 예로 볼 세그먼트 하나
상위_세개 = []                                            # 이름을 담을 빈 목록
for 번호 in 맛보기_순위표[첫_세그먼트][:3]:                # 그 세그먼트 순위표의 위 3개를 하나씩
    상위_세개.append(이름_사전[번호])                      # 번호를 이름으로 바꿔 담는다
print(f"\n순위표 만들기 → 세그먼트마다 목록 하나씩, 모두 {len(맛보기_순위표)}개")
print(f"  {첫_세그먼트} 세그먼트의 1~3위 : {상위_세개}")

print("\n※ 여기서는 전체 기록을 넣었습니다. 진짜 채점할 때는 학습 구간만 넣습니다 — 채점 구간을 보면 반칙이니까요.")


세그먼트 나누기 → 사람 300명이 세그먼트 15가지로 묶인다
  이름 예시 : ['2차전지|3', '2차전지|4', '금융|3']   ← 주력 섹터|평균 위험도 를 이어 붙인 이름

순위표 만들기 → 세그먼트마다 목록 하나씩, 모두 15개
  2차전지|3 세그먼트의 1~3위 : ['카카오', 'KODEX 미국배당다우존스', 'KODEX 국고채30년액티브']

※ 여기서는 전체 기록을 넣었습니다. 진짜 채점할 때는 학습 구간만 넣습니다 — 채점 구간을 보면 반칙이니까요.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.3 무작위로 나눠 본다  ✏️ 직접 해 보기</mark>

**3.3 에서 하는 일** — 기록을 섞어서 아무 데서나 떼는 **무작위 분할**을 직접 만듭니다.

**1. 왜 만들어 보나**
- 무작위 분할은 **가장 흔히 쓰이는 방법**입니다. 머신러닝 교재도 대부분 이것부터 가르칩니다.
- 그런데 추천에서는 이게 **반칙**입니다. 왜 반칙인지 말로만 듣지 말고 **점수로 확인**하려는 것입니다.

**2. 공정하게 견주려면 크기를 맞춰야 합니다**
- 시간순 분할은 마지막 한 달, 그러니까 **991건**을 가려 뒀습니다.
- 무작위로도 **똑같이 991건**을 가려야 비교가 됩니다. 가리는 양이 다르면 점수가 달라져도 무엇 때문인지 알 수 없습니다.


#### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 새로 나오는 문법 — <code>sample</code> · <code>iloc</code></mark>

`표.sample(frac=1.0, random_state=42)` — 표의 줄 순서를 **무작위로 섞습니다.**
- `frac=1.0` 은 「전부」라는 뜻입니다. 1.0 이면 줄을 버리지 않고 순서만 섞습니다.
- `random_state=42` 는 **섞는 방식을 고정**합니다. 이게 없으면 돌릴 때마다 결과가 달라져서 서로 점수를 비교할 수 없습니다. 42 라는 숫자 자체에 뜻은 없습니다.

`표.iloc[:100]` — **앞에서부터 100줄**을 가져옵니다. `표.iloc[100:]` 은 **101번째부터 끝까지**입니다.
- `iloc` 은 `integer`(정수)와 `location`(위치)을 합쳐 줄인 말입니다. **「정수로 위치를 지정한다」**는 뜻입니다.

아래 셀은 마음껏 고쳐 보셔도 됩니다.


In [5]:
# 이 셀에서 하는 일 — 오늘 처음 나오는 sample·iloc 을 작은 표로 먼저 해 본다
# 왜 하나 — 5줄짜리로 확인해 두면 8,086건에 그대로 써도 무슨 일이 일어나는지 안다
작은표 = pd.DataFrame({"번호": [1, 2, 3, 4, 5]})     # 1~5 가 순서대로 든 표
print("원래 순서 :", list(작은표["번호"]))   # list = 표의 한 열을 목록으로

섞은_작은표 = 작은표.sample(frac=1.0, random_state=42)   # 줄 순서를 무작위로 섞는다
print("섞은 순서 :", list(섞은_작은표["번호"]))

print("앞 2줄    :", list(섞은_작은표.iloc[:2]["번호"]))   # 앞에서부터 2줄
print("나머지    :", list(섞은_작은표.iloc[2:]["번호"]))   # 3번째부터 끝까지


원래 순서 : [1, 2, 3, 4, 5]
섞은 순서 : [2, 5, 3, 1, 4]
앞 2줄    : [2, 5]
나머지    : [3, 1, 4]


In [6]:
# 이 셀에서 하는 일 — ✏️ 빈칸 1 · 기록을 섞어서 무작위 분할을 직접 만든다
# 왜 하나 — 가장 흔히 쓰는 방법이다. 왜 반칙인지 말로만 듣지 말고 점수로 확인하려고 직접 만든다
섞은_기록 = interactions.sample(frac=1.0, random_state=42)   # 8,086건의 순서를 섞는다
print("섞기 전 첫 줄 날짜 :", interactions.iloc[0]["ts"])   # 원래는 날짜순으로 정렬돼 있다
print("섞은 뒤 첫 줄 날짜 :", 섞은_기록.iloc[0]["ts"])       # 섞었으니 아무 날짜나 온다

# 시간순 분할이 채점 구간과 같은 개수여야 공정한 비교가 됩니다.
# 시간순으로 가려 둔 표의 이름이 무엇이었는지 3.4 아래 셀에서 확인할 수 있습니다.
가릴_개수 = 991        # ← 이 숫자를 직접 세어 넣어 보세요. 힌트: len(   ) 을 쓰면 됩니다
print("가릴 개수 :", 가릴_개수, "건")               # 넣은 숫자를 바로 확인

무작위_채점구간 = 섞은_기록.iloc[:가릴_개수]      # 앞에서부터 그만큼 = 채점 구간
무작위_학습구간 = 섞은_기록.iloc[가릴_개수:]      # 나머지 = 학습 구간
print(f"\n무작위 분할 — 학습 구간 {len(무작위_학습구간):,}건 · 채점 구간 {len(무작위_채점구간):,}건")


섞기 전 첫 줄 날짜 : 2025-08-31 10:00:00
섞은 뒤 첫 줄 날짜 : 2026-08-13 22:00:00
가릴 개수 : 991 건

무작위 분할 — 학습 구간 7,095건 · 채점 구간 991건


**3. 두 분할을 나란히 놓고 봅니다**

크기는 같습니다. 다른 것은 **채점 구간이 언제 기록인가** 하나뿐입니다.


In [7]:
# 이 셀에서 하는 일 — 시간순 분할도 만들어 두 분할의 채점 구간 날짜를 나란히 본다
# 왜 하나 — 여기가 오늘의 핵심이다 — 무작위 쪽 채점 구간은 1년 내내 흩어져 있다
train, test, 기준시점 = recsys.split_by_time(interactions)   # 1주차부터 쓰던 방법
print(f"시간순 분할 — 학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건")   # 크기는 무작위 쪽과 같다
print(f"  자른 날짜 {기준시점.date()} — 이 뒤가 통째로 가려졌다")

# 빈칸 1 에 넣은 수가 이쪽과 같아야 공정한 비교입니다. 다르면 뒤 점수가 전부 달라집니다.
if 가릴_개수 != len(test):
    print(f"\n⚠ 빈칸에 넣은 {가릴_개수} 가 채점 구간 {len(test)}건과 다릅니다 — 3.3 으로 돌아가 맞춰 주세요.")
else:
    print(f"  빈칸에 넣은 수와 같습니다 ({가릴_개수}건) — 공정한 비교입니다.")
print()

# 채점 구간이 각각 언제 기록인지 봅니다. 여기가 오늘의 핵심입니다.
print("시간순 분할의 채점 구간 :", test["ts"].min().date(), "~", test["ts"].max().date())   # 마지막 한 달에 몰려 있다
print("무작위 분할의 채점 구간 :", 무작위_채점구간["ts"].min().date(), "~", 무작위_채점구간["ts"].max().date())   # 1년 내내 흩어져 있다
print()
print("→ 무작위 쪽은 1년 내내 흩어져 있습니다. 즉 학습 구간 쪽에 미래가 섞여 들어갑니다.")


시간순 분할 — 학습 구간 7,095건 · 채점 구간 991건
  자른 날짜 2026-07-30 — 이 뒤가 통째로 가려졌다
  빈칸에 넣은 수와 같습니다 (991건) — 공정한 비교입니다.

시간순 분할의 채점 구간 : 2026-07-30 ~ 2026-08-30
무작위 분할의 채점 구간 : 2025-09-18 ~ 2026-08-30

→ 무작위 쪽은 1년 내내 흩어져 있습니다. 즉 학습 구간 쪽에 미래가 섞여 들어갑니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.4 채점 함수를 하나 만들어 둔다</mark>

**3.4 에서 하는 일** — 배우는 기록과 가려 둔 기록을 받아 **Recall@10 을 돌려주는 함수**를 만듭니다.

**1. 왜 함수로 만드나**
- 오늘 채점을 **네 번** 합니다. 무작위로 한 번, 시간순으로 한 번, 그다음 기존·신규로 두 번.
- 같은 코드를 네 번 쓰는 대신 함수 하나로 묶습니다.

**2. 안에서 하는 일은 2주차와 같습니다**
1. 배우는 기록으로 세그먼트를 나누고 순위표를 만듭니다.
2. 가려 둔 기록의 사람을 한 명씩 돌면서, 그 사람의 세그먼트 순위표에서 10개를 줍니다.
3. **이미 담은 것은 정답에서 뺍니다.** 아는 것을 다시 맞히고 점수를 받으면 그것도 누수입니다.


In [8]:
# 이 셀에서 하는 일 — 채점을 함수 하나로 묶는다 (Recall@10 을 사람별로 돌려준다)
# 왜 하나 — 오늘 채점을 네 번 한다. 같은 코드를 네 번 쓰지 않으려고 함수로 만든다
def 채점하기(학습구간_기록, 채점구간_기록):
    """2주차 방식으로 추천하고 Recall@10 을 사람별로 돌려준다."""
    무리 = 세그먼트_나누기(학습구간_기록)                  # 학습 구간만 보고 세그먼트를 나눈다
    순위표 = 순위표_만들기(학습구간_기록, 무리)             # 세그먼트별 인기 순위표
    전체_인기순위 = list(학습구간_기록["item_id"].value_counts().index)   # 세그먼트를 모를 때 쓸 목록
    이미_담은것 = 학습구간_기록.groupby("user_id")["item_id"].apply(set).to_dict()   # {사람: 담은 집합}

    사람별_점수 = {}                                    # {사람: 그 사람의 Recall@10}
    for 사람, 그_사람의_기록 in 채점구간_기록.groupby("user_id"):    # 채점 대상을 한 명씩
        이미 = 이미_담은것.get(사람, set())               # 학습 구간에 담은 것 (없으면 빈 집합)
        정답 = set(그_사람의_기록["item_id"]) - 이미       # 채점 구간에서 이미 담은 것을 뺀다
        if len(정답) == 0:                               # 맞힐 것이 없는 사람은
            continue                                     # 채점에서 뺀다 (분모가 0 이 된다)
        내_순위표 = 순위표.get(무리.get(사람), 전체_인기순위)   # 세그먼트를 모르면 전체 인기순
        추천 = recsys.take(내_순위표, 이미)[:10]           # 이미 담은 것을 빼고 위에서 10개
        사람별_점수[사람] = len(정답 & set(추천)) / len(정답)   # 맞힌 개수 ÷ 정답 개수
    return 사람별_점수


def 평균내기(사람별_점수):
    """사람별 점수를 평균 낸다. 이것이 리더보드에 적는 값이다."""
    if len(사람별_점수) == 0:                            # 채점 대상이 없으면
        return 0.0                                       # 0 으로 본다
    return sum(사람별_점수.values()) / len(사람별_점수)


# 만들었으면 한 명만 넣어 봅니다. 한 사람이 어떻게 채점되는지 보고 넘어갑니다.
한_명의_채점구간 = test[test["user_id"] == "U0003"]        # 채점 구간에서 U0003 의 기록만
한_명_점수 = 채점하기(train, 한_명의_채점구간)             # 학습 구간은 그대로, 채점은 한 명만
print(f"U0003 의 채점 구간 기록 {len(한_명의_채점구간)}건 → Recall@10 = {한_명_점수['U0003']:.4f}")
print(f"  즉 맞혀야 할 것 중 {한_명_점수['U0003']:.0%} 를 추천 10개 안에 넣었다는 뜻입니다.")

print("\n채점 함수 준비 완료 — 이제 표만 바꿔 넣으면 몇 번이든 채점할 수 있습니다.")


U0003 의 채점 구간 기록 3건 → Recall@10 = 0.3333
  즉 맞혀야 할 것 중 33% 를 추천 10개 안에 넣었다는 뜻입니다.

채점 함수 준비 완료 — 이제 표만 바꿔 넣으면 몇 번이든 채점할 수 있습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.5 두 방법으로 채점해 나란히 놓는다</mark>

**3.5 에서 하는 일** — 같은 추천 방식을 무작위 분할과 시간순 분할로 각각 채점합니다.

**추천 방식은 한 글자도 바뀌지 않습니다.** 넣어 주는 표만 다릅니다.


In [9]:
# 이 셀에서 하는 일 — 무작위 분할로 채점한다
# 왜 하나 — 지난주 0.2684 와 견줘 본다. 올라가는지 내려가는지 먼저 본다
무작위_점수들 = 채점하기(무작위_학습구간, 무작위_채점구간)   # 섞어서 나눈 표를 넣는다
무작위_점수 = 평균내기(무작위_점수들)                      # 사람별 점수를 평균
print(f"무작위로 나눠 채점 — Recall@10 = {무작위_점수:.4f}   (채점 대상 {len(무작위_점수들)}명)")   # .4f = 소수 넷째 자리
print()
print("지난주 점수가 0.2684 였습니다. 올랐습니다. 기분이 좋습니다.")


무작위로 나눠 채점 — Recall@10 = 0.3150   (채점 대상 288명)

지난주 점수가 0.2684 였습니다. 올랐습니다. 기분이 좋습니다.


In [10]:
# 이 셀에서 하는 일 — 같은 추천 방식을 시간순 분할로 채점한다
# 왜 하나 — 바뀐 것은 넣어 주는 표뿐이다. 그런데도 점수가 달라진다는 것을 보려는 것이다
시간순_점수들 = 채점하기(train, test)                      # 시간으로 나눈 표를 넣는다
시간순_점수 = 평균내기(시간순_점수들)                      # 같은 함수로 평균을 낸다
print(f"시간으로 나눠 채점 — Recall@10 = {시간순_점수:.4f}   (채점 대상 {len(시간순_점수들)}명)")
print()

부풀림 = (무작위_점수 - 시간순_점수) / 시간순_점수          # (높은 값 - 낮은 값) ÷ 낮은 값
print(f"두 점수 차이 — 무작위 쪽이 {부풀림:+.1%} 부풀려져 있습니다.")
print(f"  (잰 조건 — 가린 개수 {가릴_개수}건 · 씨앗 42 · 무작위 {len(무작위_점수들)}명 · 시간순 {len(시간순_점수들)}명)")
print("  교안 슬라이드의 +17.4% 와 다르면 위 조건이 다른 것입니다. 런타임을 초기화하고 처음부터 실행해 보세요.")


시간으로 나눠 채점 — Recall@10 = 0.2684   (채점 대상 266명)

두 점수 차이 — 무작위 쪽이 +17.4% 부풀려져 있습니다.
  (잰 조건 — 가린 개수 991건 · 씨앗 42 · 무작위 288명 · 시간순 266명)
  교안 슬라이드의 +17.4% 와 다르면 위 조건이 다른 것입니다. 런타임을 초기화하고 처음부터 실행해 보세요.


**결과 — 어느 쪽을 점수판에 적어야 할까요?**

**낮은 쪽입니다.**

무작위로 나누면 8월 기록으로 배워서 3월 기록을 맞히는 일이 생깁니다. 시험 문제를 미리 본 것과 같습니다.
실제 서비스에서는 **미래를 볼 수 없으므로** 그 점수가 나오지 않습니다.

다행히 우리는 1주차부터 시간순으로 재 왔습니다. **점수판을 고칠 일은 없습니다.**

그런데 만약 처음에 흔한 방법을 골랐다면, 아홉 주 내내 부풀려진 점수를 보며 뿌듯해하고 있었을 것입니다.


In [11]:
# 이 셀에서 하는 일 — 두 점수를 한 표로 정리한다
# 왜 하나 — 어느 쪽을 점수판에 적어야 하는지 나란히 놓고 고르려는 것이다
print(f"무작위 {무작위_점수:.4f} · 시간순 {시간순_점수:.4f} — 차이 {무작위_점수 - 시간순_점수:.4f}")   # 두 값을 한 줄로 먼저 본다
비교표 = pd.DataFrame({                                   # 두 줄짜리 표를 만든다
    "Recall@10": [무작위_점수, 시간순_점수],
    "이 숫자를 믿어도 되나": ["아니오 — 미래를 보고 맞혔다", "예 — 실제 서비스와 같은 조건"],
}, index=["무작위로 나눠서", "시간으로 나눠서"]).round(4)   # 소수 네 자리까지

비교표


무작위 0.3150 · 시간순 0.2684 — 차이 0.0466


,Recall@10,이 숫자를 믿어도 되나
무작위로 나눠서,0.3150,아니오 — 미래를 보고 맞혔다
시간으로 나눠서,0.2684,예 — 실제 서비스와 같은 조건


**5. 한 가지 더 — 씨앗을 바꾸면 어떻게 될까요?**

무작위로 나눌 때 `random_state=42` 를 썼습니다. 이 숫자를 바꾸면 **다르게 섞입니다.**

같은 추천 방식, 같은 크기로 나누는데 **숫자 하나만 바꿔서** 네 번 돌려 봅니다.


In [12]:
# 이 셀에서 하는 일 — 씨앗(random_state)만 바꿔 무작위 분할을 네 번 돌린다
# 왜 하나 — 무작위는 돌릴 때마다 답이 달라진다. 그래서 씨앗을 고정해 두는 것이다
for 씨앗 in (0, 42, 777, 2026):                       # 네 가지 씨앗을 하나씩
    섞은것 = interactions.sample(frac=1.0, random_state=씨앗)   # 그 씨앗으로 섞는다
    그때_채점구간 = 섞은것.iloc[:가릴_개수]              # 앞에서부터 991건 = 채점 구간
    그때_학습구간 = 섞은것.iloc[가릴_개수:]              # 나머지 = 학습 구간
    그때_점수 = 평균내기(채점하기(그때_학습구간, 그때_채점구간))   # 같은 함수로 채점
    print(f"  random_state={씨앗:>4} → {그때_점수:.4f}")

print()
print(f"  시간으로 나누면      → {시간순_점수:.4f}   ← 씨앗이 없으니 언제나 같다")
print()
print("→ 무작위는 돌릴 때마다 답이 달라집니다. 그래서 씨앗을 고정해 두는 것입니다.")


  random_state=   0 → 0.3138

  random_state=  42 → 0.3150


  random_state= 777 → 0.3258


  random_state=2026 → 0.3115

  시간으로 나누면      → 0.2684   ← 씨앗이 없으니 언제나 같다

→ 무작위는 돌릴 때마다 답이 달라집니다. 그래서 씨앗을 고정해 두는 것입니다.


## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · 개념 — 데이터 누수와 콜드 스타트  `[PPT]`</mark>

이 파트는 슬라이드로 진행합니다. 노트북은 아래 **3.6** 부터 다시 이어집니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.6 이번 달에 처음 온 사람을 찾는다</mark>

**3.6 에서 하는 일** — 학습 구간에 기록이 없고 채점 구간에만 있는 사람을 골라냅니다.

**1. 왜 이 사람들이 문제인가**
- 2주차 방식은 **담은 기록에서 주력 섹터를 뽑습니다.** 기록이 없으면 뽑을 것이 없습니다.
- 이것을 **콜드 스타트(cold start)** 라고 부릅니다. 차가운 상태에서 시작한다는 뜻입니다.

**2. 어떻게 찾나**
- 채점 구간에 있는 사람에서 학습 구간에 있는 사람을 **빼면** 됩니다.
- 집합 빼기(`-`)는 2주차에 정답을 만들 때 썼던 것과 같습니다.


In [13]:
# 이 셀에서 하는 일 — 채점 대상을 기존 투자자와 신규 투자자로 가른다
# 왜 하나 — 학습 구간에 기록이 없는 사람이 몇 명인지 세어 본다. 이 사람들이 콜드 스타트다
학습구간_사람들 = set(train["user_id"])        # 학습 구간에 기록이 있는 사람
채점할_사람들 = set(test["user_id"])           # 채점 구간에 기록이 있는 사람
print(f"학습 구간에 기록이 있는 사람 {len(학습구간_사람들)}명 · 채점 대상 {len(채점할_사람들)}명")

신규_투자자 = 채점할_사람들 - 학습구간_사람들   # 빼기(-) = 채점에는 있는데 학습 구간에는 없던 사람
기존_투자자 = 채점할_사람들 & 학습구간_사람들   # 교집합(&) = 양쪽에 다 있는 사람
print(f"신규 투자자 {len(신규_투자자)}명 · 기존 투자자 {len(기존_투자자)}명")

# 정말 이번 달에 가입한 사람인지 users.csv 의 가입일로 확인합니다.
신규_가입일 = users[users["user_id"].isin(신규_투자자)]["joined_at"]   # isin = 목록에 드는 줄만
print()
print(f"신규 {len(신규_투자자)}명의 가입일 : {신규_가입일.min()} ~ {신규_가입일.max()}")
print(f"분할 기준 시점이 {기준시점.date()} 이니, 전원 그 뒤에 가입했습니다.")


학습 구간에 기록이 있는 사람 285명 · 채점 대상 266명
신규 투자자 15명 · 기존 투자자 251명



신규 15명의 가입일 : 2026-08-02 ~ 2026-08-23
분할 기준 시점이 2026-07-30 이니, 전원 그 뒤에 가입했습니다.


**3. 한 사람만 들여다봅니다**

숫자로만 보면 감이 안 옵니다. 신규 투자자 한 명이 실제로 어떤 상태인지 확인합니다.

`users.csv` 에는 **가입일(`joined_at`)** 칸이 있습니다. 이 사람이 정말 이번 달에 온 사람인지 그 칸으로 확인할 수 있습니다.


In [14]:
# 이 셀에서 하는 일 — 신규 투자자 한 명을 골라 상태를 들여다본다
# 왜 하나 — 숫자 15명보다 한 사람의 가입일·기록 수를 보는 편이 콜드 스타트가 무엇인지 빨리 와닿는다
기록수 = {}                                           # {사람: 채점 구간 기록 수}
for 사람 in sorted(신규_투자자):                       # 신규 15명을 한 명씩
    기록수[사람] = len(test[test["user_id"] == 사람])   # 그 사람의 채점 구간 기록 수
한_사람 = max(기록수, key=기록수.get)                   # 가장 많은 사람 (설명하기 좋다)

그_사람_정보 = users[users["user_id"] == 한_사람].iloc[0]    # users.csv 에서 그 사람 줄
학습구간_기록_한사람 = train[train["user_id"] == 한_사람]     # 그 사람의 학습 구간 기록
채점구간_기록_한사람 = test[test["user_id"] == 한_사람]       # 그 사람의 채점 구간 기록

print(f"{한_사람} 의 상태")
print(f"  가입일         : {그_사람_정보['joined_at']}   ← 분할 기준({기준시점.date()}) 뒤에 가입")
print(f"  학습 구간 기록 : {len(학습구간_기록_한사람)}건   ← 취향을 짐작할 재료가 없다")
print(f"  채점 구간 기록 : {len(채점구간_기록_한사람)}건")
print(f"  성향(설문)     : {그_사람_정보['persona']}   ← 오늘도 쓰지 않는다")


U0152 의 상태
  가입일         : 2026-08-10   ← 분할 기준(2026-07-30) 뒤에 가입
  학습 구간 기록 : 0건   ← 취향을 짐작할 재료가 없다
  채점 구간 기록 : 12건
  성향(설문)     : 성장형   ← 오늘도 쓰지 않는다


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.7 기존과 신규를 나눠서 채점한다</mark>

**3.7 에서 하는 일** — 지금까지 하나로 평균 내던 점수를 **두 무리로 나눠** 다시 봅니다.

**1. 왜 나눠 보나**
- 채점 대상 266명 중 신규는 15명입니다. **전체의 6%도 안 됩니다.**
- **왜 300명이 아니라 266명인가** — 채점은 가려 둔 마지막 한 달에 무엇을 담았는지 맞히는 일입니다. 그 한 달에 한 건도 안 담은 **34명은 맞힐 정답이 없어** 빠집니다. 300명 = 채점 266명(기존 251 + 신규 15) + 34명.
- 그래서 이 사람들 점수가 아무리 나빠도 **평균에는 거의 영향을 주지 못합니다.** 평균 뒤에 숨습니다.

**2. 어떻게 나누나**
- 새로 채점하지 않습니다. **3.5 에서 이미 사람별로 점수를 받아 뒀습니다.**
- 그 점수를 신규인지 아닌지로 갈라 평균만 따로 내면 됩니다.


In [15]:
# 이 셀에서 하는 일 — 3.5 에서 받아 둔 사람별 점수를 기존·신규로 갈라 평균만 따로 낸다
# 왜 하나 — 새로 채점하지 않는다. 전체 평균 하나에 무엇이 가려져 있었는지 보려는 것이다
기존_점수들 = {}                                  # {기존 투자자: 점수}
신규_점수들 = {}                                  # {신규 투자자: 점수}
for 사람, 점수 in 시간순_점수들.items():            # 사람별 점수를 하나씩 꺼내서
    if 사람 in 신규_투자자:                        # 신규 명단에 있으면
        신규_점수들[사람] = 점수                    # 신규 쪽에 담고
    else:                                          # 아니면
        기존_점수들[사람] = 점수                    # 기존 쪽에 담는다

print(f"전체     {len(시간순_점수들)}명  Recall@10 = {평균내기(시간순_점수들):.4f}")   # 지금까지 보던 숫자
print(f"기존     {len(기존_점수들)}명  Recall@10 = {평균내기(기존_점수들):.4f}")
print(f"신규      {len(신규_점수들)}명  Recall@10 = {평균내기(신규_점수들):.4f}")
print()
print(f"신규는 기존의 {평균내기(신규_점수들) / 평균내기(기존_점수들):.0%} 수준입니다.")   # 신규 ÷ 기존
print(f"그런데 신규는 전체의 {len(신규_점수들) / len(시간순_점수들):.0%} 뿐이라 평균을 거의 못 움직입니다.")   # 15 ÷ 266


전체     266명  Recall@10 = 0.2684
기존     251명  Recall@10 = 0.2746
신규      15명  Recall@10 = 0.1653

신규는 기존의 60% 수준입니다.
그런데 신규는 전체의 6% 뿐이라 평균을 거의 못 움직입니다.


**결과 — 평균 하나가 가리고 있었습니다**

추천 방식도, 채점 방법도 똑같습니다. **사람만 둘로 나눴을 뿐**입니다.

신규가 기존의 **60% 수준**입니다. 그런데 전체 평균은 기존 쪽 숫자와 거의 같습니다. 신규가 15명뿐이라 평균을 거의 못 움직이기 때문입니다.

**평균 하나만 보면 안 됩니다.** 누구에게 나쁜지를 나눠서 봐야 합니다.
실무에서는 신규·기존뿐 아니라 지역별, 기기별로도 나눠 봅니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.8 신규에게 무엇을 줄까  ✏️ 직접 해 보기</mark>

**3.8 에서 하는 일** — 신규 투자자에게 줄 목록을 **직접 고르고**, 점수를 다시 잰 뒤 **왜 골랐는지 한 줄** 적습니다.

**1. 정답이 없는 문제입니다**
- 후보가 둘 있습니다. 둘 다 실제로 쓰이는 방법입니다.

| 후보 | 좋은 점 | 걸리는 점 |
|---|---|---|
| **전체 인기 목록** | 많은 사람이 담은 것이라 크게 틀리지 않습니다 | 모두에게 같은 목록 — 1주차로 되돌아갑니다 |
| **위험도가 중간인 것만** | 처음 온 사람에게 레버리지를 권하지 않습니다 | 공격적인 신규 투자자에게는 답답합니다 |

**2. 점수가 떨어져도 괜찮습니다**
- 오늘의 과제는 점수를 올리는 것이 아니라 **왜 그 방법을 골랐는지 말할 수 있는 것**입니다.
- 실무에서 하는 일이 정확히 이것입니다 — 숫자 하나로 안 정해지는 것을 판단하고 근거를 남기는 일.


In [16]:
# 이 셀에서 하는 일 — ✏️ 빈칸 2 · 신규 투자자에게 줄 목록을 고른다 (전체 인기 / 위험도 중간)
# 왜 하나 — 기록이 0건이면 세그먼트를 찾을 수 없다. 사람마다 다른 목록을 못 만드니 미리 만든 목록 중에서 고른다
방법 = "인기"        # ← "인기" 또는 "중위험" 으로 바꿔 보세요

전체_인기순위 = list(train["item_id"].value_counts().index)   # 학습 구간 전체에서 많이 담긴 순
print(f"전체 인기순위 {len(전체_인기순위)}개 · 1위 {이름_사전[전체_인기순위[0]]}")   # 만든 목록을 바로 확인

중위험_순위 = []                                  # 위험도가 중간(2~3)인 종목만 모을 목록
for 번호 in 전체_인기순위:                         # 인기 순서를 그대로 유지하면서
    if 위험도_사전[번호] in (2, 3):                # 위험도가 2 또는 3 이면
        중위험_순위.append(번호)                   # 담는다
print(f"그중 위험도 2~3 만 남기면 {len(중위험_순위)}개 · 1위 {이름_사전[중위험_순위[0]]}")   # 걸러낸 뒤 몇 개 남았나

if 방법 == "인기":                                # 고른 방법에 따라 줄 목록을 정한다
    신규에게_줄_목록 = 전체_인기순위
else:
    신규에게_줄_목록 = 중위험_순위

print(f"고른 방법 : {방법} · 목록 길이 {len(신규에게_줄_목록)}개")

상위_세개 = []                                    # 이름을 담을 빈 목록
for 번호 in 신규에게_줄_목록[:3]:                  # 위에서 세 개를 하나씩
    상위_세개.append(이름_사전[번호])              # 번호를 이름으로 바꿔 담는다
print("상위 3개 :", 상위_세개)


전체 인기순위 100개 · 1위 iShares Core MSCI EAFE ETF
그중 위험도 2~3 만 남기면 48개 · 1위 iShares Core MSCI EAFE ETF
고른 방법 : 인기 · 목록 길이 100개
상위 3개 : ['iShares Core MSCI EAFE ETF', 'Vanguard FTSE Developed Markets ETF', 'TIGER 미국S&P500']


In [17]:
# 이 셀에서 하는 일 — 고른 목록으로 신규 15명만 다시 채점한다
# 왜 하나 — 점수가 오를 수도 떨어질 수도 있다. 오늘 과제는 점수가 아니라 왜 그 목록을 골랐는지다
새_신규_점수들 = {}                                # {신규 투자자: 새 점수}
for 사람 in sorted(신규_투자자):                    # 신규 15명을 한 명씩
    그_사람의_기록 = test[test["user_id"] == 사람]   # 채점 구간의 기록
    정답 = set(그_사람의_기록["item_id"])            # 학습 구간 기록이 없으니 뺄 것도 없다
    추천 = 신규에게_줄_목록[:10]                     # 고른 목록에서 위에서 10개
    새_신규_점수들[사람] = len(정답 & set(추천)) / len(정답)   # 맞힌 개수 ÷ 정답 개수

print(f"바꾸기 전 (세그먼트 방식) : {평균내기(신규_점수들):.4f}")
print(f"바꾼 뒤   ({방법} 방식)      : {평균내기(새_신규_점수들):.4f}")
print()
print("→ 올랐나요, 떨어졌나요? 어느 쪽이든 괜찮습니다. 왜 그런지가 오늘 얻어 갈 것입니다.")


바꾸기 전 (세그먼트 방식) : 0.1653
바꾼 뒤   (인기 방식)      : 0.1653

→ 올랐나요, 떨어졌나요? 어느 쪽이든 괜찮습니다. 왜 그런지가 오늘 얻어 갈 것입니다.


**3. 고르기 전에 — 신규가 실제로 무엇을 담았는지 봅니다**

「처음 온 사람에게는 안전한 것을 주자」는 생각이 자연스럽습니다.

그런데 그 생각이 맞는지 **데이터로 확인하고** 고르는 것이 낫습니다. 신규 15명이 마지막 한 달에 실제로 담은 종목을 세어 봅니다.


In [18]:
# 이 셀에서 하는 일 — 신규 15명이 실제로 담은 종목을 세어 본다
# 왜 하나 — 「처음 온 사람에게는 안전한 것을」 이라는 생각이 이 데이터에서 맞는지 확인한다
신규_담은것 = collections.Counter()                    # 종목별로 몇 명이 담았는지 셀 그릇
for 사람 in sorted(신규_투자자):                        # 신규 15명을 한 명씩
    for 번호 in test[test["user_id"] == 사람]["item_id"]:   # 그 사람이 담은 종목을 하나씩
        신규_담은것[번호] += 1                          # 그 종목을 한 번 센다

print("신규 투자자가 실제로 많이 담은 종목")
for 번호, 인원 in 신규_담은것.most_common(5):           # 많이 담긴 순으로 다섯 개
    print(f"  {이름_사전[번호]:32} {인원}명 · 위험도 {위험도_사전[번호]}")

높은위험 = 0                                            # 위험도 4 이상을 담은 건수를 셀 그릇
전체건수 = 0                                            # 전체 건수
for 번호, 인원 in 신규_담은것.items():                   # 담긴 종목을 하나씩
    전체건수 = 전체건수 + 인원
    if 위험도_사전[번호] >= 4:                          # 위험도가 4 이상이면
        높은위험 = 높은위험 + 인원

print()
print(f"신규가 담은 것 중 위험도 4 이상이 {높은위험}건 / 전체 {전체건수}건 = {높은위험/전체건수:.0%}")
print("→ 「처음 온 사람에게는 안전한 것을」 이라는 생각이 이 데이터에서는 맞지 않습니다.")


신규 투자자가 실제로 많이 담은 종목
  iShares Core MSCI EAFE ETF       5명 · 위험도 3
  Amazon                           4명 · 위험도 4
  TIGER 반도체TOP10                   4명 · 위험도 4
  에코프로비엠                           4명 · 위험도 5
  ProShares UltraPro QQQ           3명 · 위험도 5

신규가 담은 것 중 위험도 4 이상이 82건 / 전체 140건 = 59%
→ 「처음 온 사람에게는 안전한 것을」 이라는 생각이 이 데이터에서는 맞지 않습니다.


**3. 왜 그 방법을 골랐는지 한 줄 적어 주세요**

아래 칸에 직접 적으시면 됩니다. 정답이 없으니 각자 다른 답이 나오는 것이 정상입니다.

> **내가 고른 방법:**
>
> **왜 골랐나:**
>
> _(예 — 처음 온 분에게 레버리지를 권하고 싶지 않아서 중위험을 골랐다. 점수는 조금 떨어졌지만 금융 서비스에서는 이 편이 맞다고 생각한다.)_


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">3.9 오늘의 마무리  `[PPT]`</mark>

오늘은 **추천 방식을 한 글자도 고치지 않았습니다.** 그래서 점수판에 새 줄이 쌓이지 않습니다.

대신 두 가지를 얻었습니다.

1. **이 숫자를 믿어도 된다는 근거** — 무작위로 재면 0.3150 이 나오는데 그것이 거짓말이라는 것을 직접 확인했습니다.
2. **신규 15명이 0.1653 이라는 사실** — 평균 뒤에 숨어 있던 숫자입니다.

다음 주에는 **규칙을 사람이 정하지 않습니다.** 모델이 스스로 찾습니다.
그리고 그 점수를 **오늘 확인한 정직한 방법으로** 잽니다.


In [19]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 3주차는 2주차와 같은 값이다. 추천 방식을 안 고쳤으니 당연하다
recsys.record(3, "같은 추천 방식 · 정직하게 채점", 시간순_점수,
              note="추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를 보고 맞힌 값이다")

print("점수판에 3주차를 기록했습니다 — 2주차와 같은 값입니다.\n")   # \n = 한 줄 띄우기
recsys.leaderboard(upto=3)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)


레벨 3 · 같은 추천 방식 · 정직하게 채점 · Recall@10 = 0.2684
점수판에 3주차를 기록했습니다 — 2주차와 같은 값입니다.



,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,같은 추천 방식 · 정직하게 채점,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
